In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from hbn.constants import Defaults
from hbn.visualization import visualize

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

%matplotlib inline

#pio.renderers.default = 'iframe'
import plotly.io as pio
pio.renderers.default='notebook'

In [ ]:
# check models
from hbn.models.predictive_modeling import check_models

check_models(filter='*adhd-models/*')

In [ ]:
models_dict = { 'ADHD-incl-comorbidities': 'all_feature_models/2023-03-16_15-03-36-30', # incl comobordities

# loop over models
df_all = pd.DataFrame()
df_features = pd.DataFrame()
for key, value in models_dict.items():
    MODEL_DIR = os.path.join(Defaults.MODEL_DIR, value)

    df_classify = pd.read_csv(os.path.join(MODEL_DIR, 'classifier-all-phenotypic-models-performance.csv'))
    df_classify['data'] = df_classify['data'].map({'model-data': 'null', 'model-null': 'data'})
    df_classify['participant_group'] = key
    
    try:
        df_feature = pd.read_csv(os.path.join(MODEL_DIR, 'classifier-feature_importance.csv'))
        df_feature['participant_group'] = key
        df_features = pd.concat([df_features, df_feature])
    except:
        pass
    
    df_all = pd.concat([df_all, df_classify])

In [ ]:
# set plotting style
visualize.plotting_style()

In [ ]:
to_keep = ['ADHD-fullmodel', 'Depression-fullmodel', 'Anxiety-fullmodel', 'Reading Impairment-fullmodel', 'ASD-fullmodel']

df1 = df_all[(df_all['participant_group'].isin(to_keep)) & 
             (df_all['target']=='DX_01_Cat_new_binarize')]
df1['participant_group'] = df1['participant_group'].str.replace("-fullmodel","")

visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title='')
